# Streaming LLMAgent Executor — Progress Updates During Task Execution

This notebook contrasts `StreamingLLMAgentA2AExecutor` with the
non-streaming `LLMAgentA2AExecutor` from `examples/ch10.ipynb`'s
Example 4: the same Hailstone task, dispatched to two real, separately
running A2A servers, one publishing an incremental
`TaskStatusUpdateEvent` per step and one publishing a single terminal
update.

There is no client-side streaming support in `UseA2AAgentTool` yet;
dispatch there is always `ClientConfig(streaming=False)`, one
`ToolCallResult` per call, per its own docstring. So this notebook
drives the raw `a2a-sdk` client directly.

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have
Ollama installed on your local machine and have its LLM hosting
service running. To download Ollama, follow the instructions found on
this page: https://ollama.com/download. After downloading and
installing Ollama, you can start a service by opening a terminal and
running `ollama serve`.

In [2]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [3]:
model = "glm-5.3:cloud" if use_cloud else "qwen3:14b"

## The Two Servers

Both peers are the same standalone apps `examples/ch10.ipynb` uses,
`extra/a2a-from-scratch-hailstone` and its sibling
`extra/a2a-from-scratch-hailstone-streaming`, genuinely separate OS
processes, not co-routines sharing this notebook's process. The only
difference between the two apps is which executor wraps the agent:
`LLMAgentA2AExecutor` versus `StreamingLLMAgentA2AExecutor`.

In [4]:
import contextlib
import signal
from pathlib import Path
from urllib.parse import urlparse


def ensure_hailstone_server(app_dir, host, log_path, timeout=15):
    """Start a from-scratch Hailstone A2A server app if not already running.

    Returns the Popen handle if this call started the server, or None if
    it was already running (so Cleanup knows not to tear it down). Runs
    in its own session (start_new_session=True) so Cleanup can kill the
    whole process group -- terminating only the `uv run` wrapper process
    can leave the actual uvicorn process orphaned and still bound to the
    port.
    """

    def _up():
        try:
            with urllib.request.urlopen(
                f"{host}/.well-known/agent-card.json",
                timeout=1,
            ):
                pass
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        print(f"\u2713 {app_dir} A2A server already running at {host}")
        return None

    server_path = Path.cwd().parent.parent / "extra" / app_dir
    port = urlparse(host).port or 80
    env = dict(os.environ, OLLAMA_MODEL=model)
    if use_cloud:
        env["OLLAMA_HOST"] = "https://ollama.com"
    print(f"Starting {app_dir} A2A server at {host}...")
    with open(log_path, "w") as log_file:
        process = subprocess.Popen(
            [
                "uv",
                "run",
                "uvicorn",
                "main:app",
                "--host",
                "0.0.0.0",
                "--port",
                str(port),
            ],
            cwd=server_path,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=env,
        )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            print(f"\u2713 {app_dir} A2A server up at {host}")
            return process
        time.sleep(0.5)

    stop_hailstone_server(process)
    raise RuntimeError(f"A2A server did not start within {timeout}s")


def stop_hailstone_server(process):
    """Kills the whole process group started by ensure_hailstone_server()."""

    def _signal(sig):
        with contextlib.suppress(ProcessLookupError):
            os.killpg(os.getpgid(process.pid), sig)

    _signal(signal.SIGTERM)
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        _signal(signal.SIGKILL)
        process.wait(timeout=5)


streaming_process = ensure_hailstone_server(
    "a2a-from-scratch-hailstone-streaming",
    "http://127.0.0.1:9301",
    "streaming_hailstone_server.log",
)
nonstreaming_process = ensure_hailstone_server(
    "a2a-from-scratch-hailstone",
    "http://127.0.0.1:9300",
    "nonstreaming_hailstone_server.log",
)

Starting a2a-from-scratch-hailstone-streaming A2A server at http://127.0.0.1:9301...


✓ a2a-from-scratch-hailstone-streaming A2A server up at http://127.0.0.1:9301
Starting a2a-from-scratch-hailstone A2A server at http://127.0.0.1:9300...


✓ a2a-from-scratch-hailstone A2A server up at http://127.0.0.1:9300


## Dispatching with a Raw Streaming Client

`A2AAgentSpec.from_url()` still does discovery, the same as
`examples/ch10.ipynb`. Everything after that is hand-rolled: build an
SDK client with `ClientConfig(streaming=...)`, send one message, and
print every chunk `client.send_message()` yields as it arrives, rather
than collapsing to a single final result the way `UseA2AAgentTool`
does.

In [5]:
import httpx
from a2a.client import ClientConfig, create_client
from a2a.helpers import new_text_message
from a2a.types import Role as A2ARole
from a2a.types import SendMessageRequest, TaskState

from llm_agents_from_scratch.a2a import A2AAgentSpec
from llm_agents_from_scratch.a2a.client.utils import a2a_parts_text

streaming_spec = await A2AAgentSpec.from_url(
    "http://127.0.0.1:9301",
    timeout=120.0,
)
nonstreaming_spec = await A2AAgentSpec.from_url(
    "http://127.0.0.1:9300",
    timeout=120.0,
)


async def dispatch_and_print(spec, streaming, task_text):
    """Send one task to a peer, printing each StreamResponse chunk live."""
    async with httpx.AsyncClient(timeout=120.0) as httpx_client:
        client = await create_client(
            agent=spec.agent_card,
            client_config=ClientConfig(
                streaming=streaming,
                httpx_client=httpx_client,
            ),
        )
        n_chunks = 0
        try:
            message = new_text_message(text=task_text, role=A2ARole.ROLE_USER)
            request = SendMessageRequest(message=message)
            async for chunk in client.send_message(request):
                n_chunks += 1
                kind = chunk.WhichOneof("payload")
                if kind == "status_update":
                    state = TaskState.Name(chunk.status_update.status.state)
                    has_message = chunk.status_update.status.HasField("message")
                    text = (
                        a2a_parts_text(chunk.status_update.status.message.parts)
                        if has_message
                        else ""
                    )
                    print(f"  [{n_chunks}] {state}: {text[:90]!r}")
                elif kind == "task":
                    state = TaskState.Name(chunk.task.status.state)
                    print(f"  [{n_chunks}] task created, state={state}")
                elif kind == "artifact_update":
                    text = a2a_parts_text(chunk.artifact_update.artifact.parts)
                    print(f"  [{n_chunks}] artifact_update: {text[:90]!r}")
        finally:
            await client.close()
    print(f"  total chunks: {n_chunks}")
    return n_chunks

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Example — The Same Task, Two Executors

6 is a short enough Hailstone sequence to run in seconds, but long
enough (8 steps) to show several distinct status updates land before
completion.

In [6]:
task_text = "Compute the hailstone sequence starting at 6, until it reaches 1."

print("=== Streaming ===")
streaming_chunks = await dispatch_and_print(streaming_spec, True, task_text)

print()
print("=== Non-streaming ===")
nonstreaming_chunks = await dispatch_and_print(
    nonstreaming_spec,
    False,
    task_text,
)

=== Streaming ===

  [1] task created, state=TASK_STATE_SUBMITTED

  [2] TASK_STATE_SUBMITTED: ''

  [3] TASK_STATE_WORKING: ''

  [4] TASK_STATE_WORKING: 'Compute the hailstone sequence starting at 6, until it reaches 1.'

...

  [21] TASK_STATE_COMPLETED: ''

  total chunks: 21

=== Non-streaming ===

  [1] task created, state=TASK_STATE_COMPLETED

  total chunks: 1

The streaming peer's chunk count includes two updates per step
(the planned instruction, then the step's result), plus the initial
task-submitted chunk, a trailing artifact, and the final completed
status; a modest 8-step sequence already produces around 20 chunks.
The non-streaming peer produces exactly one, the same shape
`UseA2AAgentTool` already gets today.

## Cleanup

In [7]:
if streaming_process is not None:
    stop_hailstone_server(streaming_process)
if nonstreaming_process is not None:
    stop_hailstone_server(nonstreaming_process)